[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fiit-ba/zneus-2026/blob/main/labs/week_03_backprop_and_optimizers/task_3_optimizers.ipynb)

# Week 3 · Task 3: Optimizers from scratch

Forward pass (week 2) and backward pass (Task 2) are done. The last missing piece of our little framework is the
**optimizer**: the rule that turns gradients into parameter updates. In this notebook you implement SGD with
momentum, RMSprop and Adam, verify each of them against `torch.optim`, and compare all four on a small
classification problem while tracking the runs in Weights & Biases.

## What you will do
1. **The framework so far** (given) - `Linear`, activations, losses and `Model`, now with complete forward and backward passes.
2. **Data** (given) - the Flower dataset, mini-batching and a decision-boundary plot.
3. **Optimizers** - implement `SGDMomentum`, `RMSprop` and `Adam`, and verify each against `torch.optim` after 3 steps.
4. **Comparison** - train the same MLP with all four optimizers, log every run to W&B, plot the loss curves and the best decision boundary.
5. **Questions** - check your understanding.

## How to work through this notebook
- Every place that needs your input is marked with a `# TODO` comment and/or `...`. **Replace every `...` with your own code.**
- Written answers go into the markdown cells marked _Your answer here._ (double-click the cell to edit it).
- Every optimizer has a **verification cell** that compares it with the corresponding `torch.optim` optimizer and
  raises an `AssertionError` on a `MISMATCH`. You are done with an optimizer when its verification prints `OK`. If
  you run the cells one by one, the comparison in section 4 skips optimizers that have not passed yet; the hand-in
  requires all four.
- Run the cells **in order**: later cells depend on classes defined earlier.
- Our framework uses `torch.float64` tensors and **no autograd**; `torch.optim` and autograd appear only as references.

Working in Colab? Replace `fiit-ba` in the URL with your GitHub username to open the copy in your fork, and run the
setup cell below.

_Credit: this notebook is a PyTorch port of the NSIETE (FIIT STU) numpy lab "Task 3 - optimization"._

In [ ]:
# Colab setup (no-op locally)
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wandb"], check=True)

In [ ]:
%matplotlib inline
import math
import os
from collections import OrderedDict

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import wandb
from tqdm.auto import tqdm

torch.set_printoptions(precision=4, sci_mode=False)

In [ ]:
# --- Configuration ---
SEED = 42
BATCH_SIZE = 32
EPOCHS = 500            # per optimizer; one run takes a few seconds on a laptop CPU
M_SAMPLES = 512         # size of the synthetic Flower dataset
NOISE = 0.3             # noise of the dataset
SUBSET = None           # e.g. 128 while debugging, None = all M_SAMPLES samples
DTYPE = torch.float64   # our framework works in double precision, without autograd
WANDB_PROJECT = "zneus-2026"
WANDB_GROUP = "week03-optimizers"
# os.environ["WANDB_MODE"] = "offline"   # uncomment if you do not have a W&B account yet (see below)

torch.manual_seed(SEED)

### Weights & Biases

Every experiment that trains a model in this course is tracked in [Weights & Biases](https://wandb.ai). Log in
**once** with `wandb login` in a terminal (or `wandb.login()` in a cell) and paste the API key from
[wandb.ai/authorize](https://wandb.ai/authorize). No account yet? Set `WANDB_MODE=offline` (see the commented line
in the configuration cell): the runs are then stored locally in a `wandb/` folder and can be uploaded later with
`wandb sync`. The comparison in section 4 creates one run per optimizer in project `zneus-2026`, group
`week03-optimizers`. Make the project **Public** or **Open** in its settings and hand in the link to the project (or
the group). Never hard-code your W&B entity or API key in the notebook.

## 1. The framework so far (given)

Everything from Task 2, complete. Two small additions to `Linear` are the interface an optimizer uses to talk to a
layer:

- `get_optimizer_context()` returns `[[W, dW], [b, db]]`: the parameters and their current gradients;
- `set_optimizer_context([W, b])` writes the updated parameters back.

Reminder of the conventions: an input batch `X` has shape `(n_features, m)`, `W` has shape `(out, in)`, `b` has
shape `(out, 1)`; the gradient flowing between layers is per-sample and `Linear.backward` applies the `1/m` when it
forms `dW` and `db`.

In [ ]:
class Module:
    """Minimal base class of our little framework: forward, backward and named sub-modules."""

    def __init__(self) -> None:
        self.modules: OrderedDict[str, "Module"] = OrderedDict()

    def add_module(self, module: "Module", name: str) -> None:
        if not name or "." in name:
            raise KeyError(f"invalid module name {name!r}")
        if name in self.modules:
            raise KeyError(f"module {name!r} already exists")
        self.modules[name] = module

    def forward(self, *args, **kwargs):
        raise NotImplementedError

    def backward(self, *args, **kwargs):
        raise NotImplementedError

    def __call__(self, *args, **kwargs):
        return self.forward(*args, **kwargs)

    def __repr__(self) -> str:
        return f"{type(self).__name__}()"


class Linear(Module):
    """Fully connected layer, column convention: Z = W @ A_prev + b with A_prev of shape (in_features, m)."""

    def __init__(self, in_features: int, out_features: int) -> None:
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.W = torch.randn(out_features, in_features, dtype=DTYPE)
        self.b = torch.zeros(out_features, 1, dtype=DTYPE)
        self.dW = torch.zeros_like(self.W)
        self.db = torch.zeros_like(self.b)

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        self.fw_inputs = input
        self.m = input.shape[1]
        return self.W @ input + self.b

    def backward(self, dZ: torch.Tensor) -> torch.Tensor:
        self.dW = (1.0 / self.m) * (dZ @ self.fw_inputs.T)
        self.db = (1.0 / self.m) * dZ.sum(dim=1, keepdim=True)
        return self.W.T @ dZ

    def get_optimizer_context(self) -> list[list[torch.Tensor]]:
        """Hand the parameters and their gradients to an optimizer: [[W, dW], [b, db]]."""
        return [[self.W, self.dW], [self.b, self.db]]

    def set_optimizer_context(self, params: list[torch.Tensor]) -> None:
        """Receive the updated parameters [W, b] back from the optimizer."""
        self.W, self.b = params

    def __repr__(self) -> str:
        return f"Linear(in_features={self.in_features}, out_features={self.out_features})"


class Sigmoid(Module):
    def forward(self, input: torch.Tensor) -> torch.Tensor:
        self.fw_input = input
        return 1.0 / (1.0 + torch.exp(-input))

    def backward(self, dA: torch.Tensor) -> torch.Tensor:
        a = self.forward(self.fw_input)
        return dA * a * (1.0 - a)


class Tanh(Module):
    def forward(self, input: torch.Tensor) -> torch.Tensor:
        self.fw_input = input
        return torch.tanh(input)

    def backward(self, dA: torch.Tensor) -> torch.Tensor:
        a = self.forward(self.fw_input)
        return dA * (1.0 - a ** 2)


class ReLU(Module):
    def forward(self, input: torch.Tensor) -> torch.Tensor:
        self.fw_input = input
        return torch.clamp(input, min=0.0)

    def backward(self, dA: torch.Tensor) -> torch.Tensor:
        return dA * (self.fw_input > 0).to(dA.dtype)

**Loss functions.** Unlike in Task 2, `forward` now returns the **cost** of the batch directly: the mean of the
per-sample losses (what PyTorch's losses return with their default `reduction="mean"`). `backward` returns the
per-sample gradient $\partial L_i / \partial \hat y_i$; the $1/m$ is applied inside `Linear.backward`, exactly as
in Task 2.

In [ ]:
class _Loss(Module):
    """Base class of the losses: forward returns the cost (mean over the batch), backward the per-sample gradient."""


class MSELoss(_Loss):
    """Mean squared error of the batch."""

    def forward(self, input: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        return ((target - input) ** 2).mean()

    def backward(self, input: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        return -2.0 * (target - input)


class BCELoss(_Loss):
    """Binary cross-entropy for predicted probabilities in (0, 1) and 0/1 targets, averaged over the batch."""

    EPS = 1e-12   # predictions are clamped to [EPS, 1 - EPS] so that log(0) and x/0 never happen

    def forward(self, input: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        y_hat = torch.clamp(input, self.EPS, 1.0 - self.EPS)
        return (-(target * torch.log(y_hat) + (1.0 - target) * torch.log(1.0 - y_hat))).mean()

    def backward(self, input: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        y_hat = torch.clamp(input, self.EPS, 1.0 - self.EPS)
        return -(target / y_hat - (1.0 - target) / (1.0 - y_hat))

In [ ]:
class Model(Module):
    """A sequential container: forward runs the modules in insertion order, backward in reverse order."""

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        for name, module in self.modules.items():
            input = module(input)
        return input

    def backward(self, dZ: torch.Tensor) -> torch.Tensor:
        for name, module in reversed(self.modules.items()):
            dZ = module.backward(dZ)
        return dZ

    def __repr__(self) -> str:
        body = "\n".join(f"  ({name}): {module}" for name, module in self.modules.items())
        return f"Model(\n{body}\n)"

A quick sanity check that the given framework computes correct gradients (nothing to do here): a tiny network's
`dW`, `db` against autograd applied to PyTorch's own `binary_cross_entropy`. As in Task 2, `OK` means every element
agrees to within `1e-10` (absolute tolerance, no relative tolerance).

In [ ]:
torch.manual_seed(0)
net = Model()
net.add_module(Linear(2, 3), "Dense_1")
net.add_module(Tanh(), "Tanh_1")
net.add_module(Linear(3, 1), "Dense_2")
net.add_module(Sigmoid(), "Sigmoid")
Xs = torch.randn(2, 6, dtype=DTYPE)
Ys = torch.randint(0, 2, (1, 6)).to(DTYPE)

criterion = BCELoss()
Y_hat = net(Xs)
dY_hat = criterion.backward(Y_hat, Ys)
net.backward(dY_hat)

W1 = net.modules["Dense_1"].W.clone().requires_grad_(True)
b1 = net.modules["Dense_1"].b.clone().requires_grad_(True)
W2 = net.modules["Dense_2"].W.clone().requires_grad_(True)
b2 = net.modules["Dense_2"].b.clone().requires_grad_(True)
A1 = torch.tanh(W1 @ Xs + b1)
Y_hat_ref = torch.sigmoid(W2 @ A1 + b2)
F.binary_cross_entropy(Y_hat_ref, Ys, reduction="mean").backward()

print(f"cost = {criterion(Y_hat, Ys).item():.4f}")
comparisons = [
    ("Dense_1.dW", net.modules["Dense_1"].dW, W1.grad),
    ("Dense_1.db", net.modules["Dense_1"].db, b1.grad),
    ("Dense_2.dW", net.modules["Dense_2"].dW, W2.grad),
    ("Dense_2.db", net.modules["Dense_2"].db, b2.grad),
]
for name, mine, ref in comparisons:
    ok = torch.allclose(mine, ref, atol=1e-10, rtol=0.0)
    max_diff = (mine - ref).abs().max().item()
    print(f"{'OK        ' if ok else 'MISMATCH  '}{name:<12s} max |diff| = {max_diff:.1e}")
    assert ok, f"the given framework's {name} disagrees with autograd; did you change Linear.backward?"

## 2. Data: the Flower dataset (given)

Two interleaved petal-shaped classes ($r = \sin 4t$ in polar coordinates, one class per half turn) with Gaussian
noise. It is not linearly separable and small enough to train in seconds, which makes it a good playground for
comparing optimizers. The data comes in our column convention: `X` has shape `(2, m)` and `Y` has shape `(1, m)`.

The samples are shuffled once, right after generation, and `make_batches` cuts them into a list of
`(X_batch, Y_batch)` mini-batches; the same mini-batches are reused in every epoch (a real `DataLoader` reshuffles
every epoch, we keep it simple here).

In [ ]:
def dataset_flower(m: int = 512, noise: float = 0.0) -> tuple[torch.Tensor, torch.Tensor]:
    """Two petal-shaped classes. Returns X of shape (2, m) and Y of shape (1, m), float64 (column convention)."""
    if m % 2:
        raise ValueError("m must be even (two classes of m/2 samples)")
    X = torch.zeros(m, 2, dtype=DTYPE)
    Y = torch.zeros(m, 1, dtype=DTYPE)
    half = m // 2
    for j in range(2):
        rows = slice(half * j, half * (j + 1))
        t = torch.linspace(j * math.pi, (j + 1) * math.pi, half, dtype=DTYPE) + torch.randn(half, dtype=DTYPE) * noise
        r = torch.sin(4 * t) + torch.randn(half, dtype=DTYPE) * noise
        X[rows] = torch.stack([r * torch.sin(t), r * torch.cos(t)], dim=1)
        Y[rows] = j
    return X.T.contiguous(), Y.T.contiguous()


def make_batches(dataset: tuple[torch.Tensor, torch.Tensor], batch_size: int) -> list[tuple[torch.Tensor, torch.Tensor]]:
    """Cut (X, Y) with X of shape (n_x, m) into a list of (X_batch, Y_batch) in order; batch_size <= 0 means one batch."""
    X, Y = dataset
    m = X.shape[1]
    if batch_size <= 0:
        batch_size = m
    return [(X[:, i:i + batch_size], Y[:, i:i + batch_size]) for i in range(0, m, batch_size)]


def draw_dataset(X: torch.Tensor, Y: torch.Tensor, title: str = "Flower dataset") -> None:
    plt.figure(figsize=(5, 5))
    plt.scatter(X[0], X[1], c=Y[0], cmap="RdBu", edgecolors="k", s=20)
    plt.gca().set_aspect("equal")
    plt.xlabel("$x_1$")
    plt.ylabel("$x_2$")
    plt.title(title)
    plt.show()


def draw_decision_boundary(X: torch.Tensor, Y: torch.Tensor, model: Model, title: str = "Decision boundary",
                           size: float = 6, h: float = 0.01) -> None:
    """Colour the plane by the model's predicted P(y = 1) and overlay the data."""
    pad = 0.5
    x1_lo = X[0].min().item() - pad
    x1_hi = X[0].max().item() + pad
    x2_lo = X[1].min().item() - pad
    x2_hi = X[1].max().item() + pad
    x1 = torch.arange(x1_lo, x1_hi, h, dtype=DTYPE)
    x2 = torch.arange(x2_lo, x2_hi, h, dtype=DTYPE)
    G1, G2 = torch.meshgrid(x1, x2, indexing="xy")
    grid = torch.stack([G1.reshape(-1), G2.reshape(-1)])       # (2, n_grid): the same column convention as X
    P = model(grid).reshape(G1.shape)                           # predicted P(y = 1) at every grid point
    plt.figure(figsize=(size, size))
    plt.contourf(G1, G2, P, levels=20, cmap="RdYlBu", alpha=0.85)
    plt.colorbar(label="predicted $P(y=1)$", shrink=0.8)
    plt.contour(G1, G2, P, levels=[0.5], colors="k", linewidths=1)
    plt.scatter(X[0], X[1], c=Y[0], cmap="RdBu", edgecolors="k", s=20)
    plt.gca().set_aspect("equal")
    plt.xlabel("$x_1$")
    plt.ylabel("$x_2$")
    plt.title(title)
    plt.show()

In [ ]:
torch.manual_seed(SEED)
X, Y = dataset_flower(m=M_SAMPLES, noise=NOISE)
perm = torch.randperm(X.shape[1])          # shuffle once: dataset_flower generates the two classes one after the other
X, Y = X[:, perm], Y[:, perm]
if SUBSET is not None:                     # a smaller dataset while debugging
    X, Y = X[:, :SUBSET], Y[:, :SUBSET]
dataset = make_batches((X, Y), BATCH_SIZE)

print(f"X: {tuple(X.shape)}  Y: {tuple(Y.shape)}  ->  {len(dataset)} mini-batches of up to {BATCH_SIZE} samples")
print("first mini-batch:", tuple(dataset[0][0].shape), tuple(dataset[0][1].shape))
draw_dataset(X, Y)

## 3. Optimizers

An optimizer owns the **update rule**. Its `step(model)` visits every layer that has parameters, asks for
`[[W, dW], [b, db]]` via `get_optimizer_context()`, computes new values and writes them back with
`set_optimizer_context([W, b])`. Optimizers with memory (all but plain SGD) keep their **state per layer** in the
dictionary `self.context[name]` (`name` is the layer's name in `model.modules`), created on the first visit of that
layer with tensors of the same shape as `W` and `b`.

Below, $\theta$ stands for either `W` or `b`, $g$ for its gradient (`dW` or `db`) and $\eta$ for the learning rate
`lr`. The rules follow the exact conventions of `torch.optim`, so that the verification cells pass.

**SGD** (given): a step against the gradient.

$$\theta \leftarrow \theta - \eta\, g$$

**SGD with momentum** (`beta` $= \beta$, typically $0.9$). Keep a velocity $v$ (initially $0$) that accumulates
gradients, and step along the velocity. Note the PyTorch flavour: the new gradient is added with weight $1$, not
$1 - \beta$.

$$v \leftarrow \beta\, v + g, \qquad \theta \leftarrow \theta - \eta\, v$$

**RMSprop** (`alpha` $= \alpha$, `eps` $= \varepsilon = 10^{-8}$). Keep a running average $s$ of the *squared*
gradients (initially $0$) and divide the step by its square root: parameters with consistently large gradients get
smaller steps, parameters with tiny gradients get larger ones.

$$s \leftarrow \alpha\, s + (1 - \alpha)\, g^2, \qquad \theta \leftarrow \theta - \eta\, \frac{g}{\sqrt{s} + \varepsilon}$$

**Adam** (`beta1` $= \beta_1$, `beta2` $= \beta_2$, `eps` $= \varepsilon = 10^{-8}$). Momentum and RMSprop combined,
plus a **bias correction** that compensates for $m$ and $v$ starting at $0$. The step counter $t$ is one integer for
the whole optimizer, incremented once per `step()` call (before the loop over the layers).

$$t \leftarrow t + 1, \qquad
  m \leftarrow \beta_1 m + (1 - \beta_1)\, g, \qquad
  v \leftarrow \beta_2 v + (1 - \beta_2)\, g^2$$

$$\hat m = \frac{m}{1 - \beta_1^t}, \qquad
  \hat v = \frac{v}{1 - \beta_2^t}, \qquad
  \theta \leftarrow \theta - \eta\, \frac{\hat m}{\sqrt{\hat v} + \varepsilon}$$

All operations are elementwise (`*`, `**`, `torch.sqrt`, `/`). Every rule is applied to `W` with `dW` and,
identically, to `b` with `db`.

In [ ]:
class Optimizer:
    """Base class: `step(model)` updates the parameters of every layer of the model with the optimizer's rule."""

    def step(self, model: Model) -> None:
        raise NotImplementedError

    def hyperparameters(self) -> dict[str, float]:
        """Scalar attributes (lr, beta, ...) for logging, used as W&B config."""
        return {k: v for k, v in vars(self).items() if isinstance(v, (int, float)) and k != "t"}

    def __repr__(self) -> str:
        params = ", ".join(f"{k}={v}" for k, v in self.hyperparameters().items())
        return f"{type(self).__name__}({params})"


class SGD(Optimizer):
    """Plain stochastic gradient descent: theta <- theta - lr * grad."""

    def __init__(self, lr: float) -> None:
        super().__init__()
        self.lr = lr

    def step(self, model: Model) -> None:
        for name, layer in model.modules.items():
            if hasattr(layer, "get_optimizer_context"):
                [[W, dW], [b, db]] = layer.get_optimizer_context()
                W = W - self.lr * dW
                b = b - self.lr * db
                layer.set_optimizer_context([W, b])

**Verification (given).** `check_against_torch` builds a tiny two-layer model with fixed weights, copies the
parameters into `torch.nn.Parameter`s, and then, for 3 steps, assigns the **same made-up gradients** to your
optimizer (via `layer.dW`, `layer.db`) and to a `torch.optim` optimizer (via `param.grad`). After 3 steps both
sets of parameters must agree in every element to within $10^{-10}$ (absolute tolerance, no relative tolerance),
otherwise the cell raises an `AssertionError`. Every optimizer
gets one call; the result is remembered in `VERIFIED` so that the comparison in section 4 knows which optimizers are
ready.

In [ ]:
VERIFIED: dict[str, bool] = {}


def check_against_torch(name: str, make_mine, make_torch, steps: int = 3) -> None:
    """Compare your optimizer with a torch.optim optimizer on identical parameters and identical fake gradients.

    make_mine() -> your Optimizer;  make_torch(list_of_parameters) -> a torch.optim.Optimizer.
    """
    torch.manual_seed(0)
    model = Model()
    model.add_module(Linear(3, 2), "Dense_1")
    model.add_module(Tanh(), "Tanh_1")
    model.add_module(Linear(2, 1), "Dense_2")
    layers = [layer for layer in model.modules.values() if isinstance(layer, Linear)]
    torch_params = [nn.Parameter(t.clone()) for layer in layers for t in (layer.W, layer.b)]

    ok, max_diff, reason = False, float("nan"), ""
    try:
        my_opt = make_mine()
        torch_opt = make_torch(torch_params)
        for _ in range(steps):
            grads = [torch.randn_like(p) for p in torch_params]           # the same fake gradients for both
            for layer, gW, gb in zip(layers, grads[0::2], grads[1::2]):
                layer.dW, layer.db = gW.clone(), gb.clone()
            for p, g in zip(torch_params, grads):
                p.grad = g.clone()
            my_opt.step(model)
            torch_opt.step()
        mine = [t for layer in layers for t in (layer.W, layer.b)]
        max_diff = max((a - p.detach()).abs().max().item() for a, p in zip(mine, torch_params))
        ok = all(torch.allclose(a, p.detach(), atol=1e-10, rtol=0.0) for a, p in zip(mine, torch_params))
    except Exception as e:  # noqa: BLE001 - a half-implemented optimizer should report, not crash the notebook
        reason = f"  ({type(e).__name__}: {e})"
    VERIFIED[name] = ok
    print(f"{'OK        ' if ok else 'MISMATCH  '}{name:<12s} vs torch.optim after {steps} steps: max |diff| = {max_diff:.1e}{reason}")
    assert ok, f"{name} does not match torch.optim after {steps} steps (max |diff| = {max_diff:.1e}){reason}"


check_against_torch("SGD", lambda: SGD(lr=0.1), lambda params: torch.optim.SGD(params, lr=0.1))

**3.1 Implement `SGDMomentum`.** Store the hyper-parameters in the constructor, create the empty state dictionary
`self.context`, and fill in the three marked steps in `step`: create the layer's state on the first visit, update
the velocities, take the step. Keep the loop skeleton as it is.

Hints: `torch.zeros_like(W)` creates the initial velocity; `name not in self.context` detects the first visit.

In [ ]:
class SGDMomentum(Optimizer):
    """SGD with momentum: v <- beta * v + grad;  theta <- theta - lr * v   (torch.optim.SGD(momentum=beta))."""

    def __init__(self, lr: float, beta: float = 0.9) -> None:
        super().__init__()
        # TODO: store lr and beta, and create the empty per-layer state dictionary self.context
        self.lr = ...
        self.beta = ...
        self.context = ...

    def step(self, model: Model) -> None:
        for name, layer in model.modules.items():
            if hasattr(layer, "get_optimizer_context"):
                [[W, dW], [b, db]] = layer.get_optimizer_context()
                # TODO:
                #  (1) first visit of this layer: self.context[name] = {"vW": zeros like W, "vb": zeros like b}
                #  (2) update the velocities:     v <- beta * v + grad          (for W and for b)
                #  (3) take the step:             W <- W - lr * vW,  b <- b - lr * vb
                if name not in self.context:
                    self.context[name] = {"vW": ..., "vb": ...}
                ctx = self.context[name]
                ctx["vW"] = ...
                ctx["vb"] = ...
                W = ...
                b = ...
                layer.set_optimizer_context([W, b])

In [ ]:
check_against_torch("SGDMomentum",
                    lambda: SGDMomentum(lr=0.1, beta=0.9),
                    lambda params: torch.optim.SGD(params, lr=0.1, momentum=0.9))

**3.2 Implement `RMSprop`.** Same structure; the state is the running average of the squared gradients (`sW`, `sb`).
`torch.optim.RMSprop` adds `eps` **after** the square root, `sqrt(s) + eps`, and so does the formula above; write
it that way. (With `eps = 1e-8` the verification cell cannot tell `sqrt(s) + eps` from `sqrt(s + eps)`: the
difference is far below its tolerance. This is about matching the convention, not about passing the check.)

In [ ]:
class RMSprop(Optimizer):
    """RMSprop: s <- alpha * s + (1 - alpha) * grad^2;  theta <- theta - lr * grad / (sqrt(s) + eps)."""

    def __init__(self, lr: float, alpha: float = 0.9, eps: float = 1e-8) -> None:
        super().__init__()
        # TODO: store lr, alpha and eps, and create the empty per-layer state dictionary self.context
        self.lr = ...
        self.alpha = ...
        self.eps = ...
        self.context = ...

    def step(self, model: Model) -> None:
        for name, layer in model.modules.items():
            if hasattr(layer, "get_optimizer_context"):
                [[W, dW], [b, db]] = layer.get_optimizer_context()
                # TODO:
                #  (1) first visit of this layer: self.context[name] = {"sW": zeros like W, "sb": zeros like b}
                #  (2) update the running averages of the squared gradients:  s <- alpha * s + (1 - alpha) * grad^2
                #  (3) take the step:  W <- W - lr * dW / (sqrt(sW) + eps),  b <- b - lr * db / (sqrt(sb) + eps)
                if name not in self.context:
                    self.context[name] = {"sW": ..., "sb": ...}
                ctx = self.context[name]
                ctx["sW"] = ...
                ctx["sb"] = ...
                W = ...
                b = ...
                layer.set_optimizer_context([W, b])

In [ ]:
check_against_torch("RMSprop",
                    lambda: RMSprop(lr=0.01, alpha=0.9, eps=1e-8),
                    lambda params: torch.optim.RMSprop(params, lr=0.01, alpha=0.9, eps=1e-8))

**3.3 Implement `Adam`.** Two running averages per parameter (`mW`, `vW`, `mb`, `vb`), a step counter `self.t`
that belongs to the optimizer (not to a layer) and is incremented once at the beginning of `step`, and the bias
correction before the update.

Hints: `self.beta1 ** self.t` is the power you need in the bias correction. Do not store the corrected `m_hat`,
`v_hat` back into the state; the state keeps the uncorrected averages.

In [ ]:
class Adam(Optimizer):
    """Adam: momentum + RMSprop with bias correction (torch.optim.Adam)."""

    def __init__(self, lr: float, beta1: float = 0.9, beta2: float = 0.999, eps: float = 1e-8) -> None:
        super().__init__()
        # TODO: store lr, beta1, beta2 and eps, create the step counter self.t = 0 and the empty state dict self.context
        self.lr = ...
        self.beta1 = ...
        self.beta2 = ...
        self.eps = ...
        self.t = ...
        self.context = ...

    def step(self, model: Model) -> None:
        # TODO: increment the step counter (once per step, before visiting the layers)
        ...
        for name, layer in model.modules.items():
            if hasattr(layer, "get_optimizer_context"):
                [[W, dW], [b, db]] = layer.get_optimizer_context()
                # TODO:
                #  (1) first visit of this layer: state with "mW", "vW" (zeros like W) and "mb", "vb" (zeros like b)
                #  (2) update the averages:  m <- beta1 * m + (1 - beta1) * grad;  v <- beta2 * v + (1 - beta2) * grad^2
                #  (3) bias correction:      m_hat = m / (1 - beta1^t);  v_hat = v / (1 - beta2^t)
                #  (4) take the step:        theta <- theta - lr * m_hat / (sqrt(v_hat) + eps)     (for W and for b)
                if name not in self.context:
                    self.context[name] = {"mW": ..., "vW": ..., "mb": ..., "vb": ...}
                ctx = self.context[name]
                ctx["mW"] = ...
                ctx["vW"] = ...
                ctx["mb"] = ...
                ctx["vb"] = ...
                mW_hat = ...
                vW_hat = ...
                mb_hat = ...
                vb_hat = ...
                W = ...
                b = ...
                layer.set_optimizer_context([W, b])

In [ ]:
check_against_torch("Adam",
                    lambda: Adam(lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8),
                    lambda params: torch.optim.Adam(params, lr=0.01, betas=(0.9, 0.999), eps=1e-8))

## 4. Comparing the optimizers

`create_model` builds the MLP of the original lab (2-3-4-5-1, Tanh hidden activations, Sigmoid output) and `fit`
runs the training loop you already know from the lecture: forward, loss, backward, step, for every mini-batch and
every epoch. It returns the mean training loss of each epoch and logs it to W&B as `train/loss`.

In [ ]:
def create_model() -> Model:
    """The MLP of the original lab: 2-3-4-5-1 with Tanh hidden activations and a Sigmoid output."""
    mlp = Model()
    mlp.add_module(Linear(2, 3), "Dense_1")
    mlp.add_module(Tanh(), "Tanh_1")
    mlp.add_module(Linear(3, 4), "Dense_2")
    mlp.add_module(Tanh(), "Tanh_2")
    mlp.add_module(Linear(4, 5), "Dense_3")
    mlp.add_module(Tanh(), "Tanh_3")
    mlp.add_module(Linear(5, 1), "Dense_4_out")
    mlp.add_module(Sigmoid(), "Sigmoid")
    return mlp


def fit(model: Model, optimizer: Optimizer, dataset: list[tuple[torch.Tensor, torch.Tensor]], criterion: _Loss,
        num_epochs: int, wandb_run=None, desc: str = "") -> list[float]:
    """Train for num_epochs over the given mini-batches; returns the mean training loss of every epoch."""
    epoch_losses: list[float] = []
    for epoch in tqdm(range(num_epochs), desc=desc or "training", leave=False):
        batch_losses: list[float] = []
        for Xb, Yb in dataset:
            Y_hat = model(Xb)                                  # forward pass
            loss = criterion(Y_hat, Yb)                        # cost of the mini-batch (mean over its samples)
            dY_hat = criterion.backward(Y_hat, Yb)             # gradient of the cost with respect to the output
            model.backward(dY_hat)                             # backward pass: fills dW, db of every layer
            optimizer.step(model)                              # parameter update
            batch_losses.append(loss.item())
        epoch_loss = sum(batch_losses) / len(batch_losses)
        epoch_losses.append(epoch_loss)
        if wandb_run is not None:
            wandb_run.log({"train/loss": epoch_loss}, step=epoch)
    return epoch_losses

**The comparison (given).** Every optimizer trains its own copy of the model, starting from **identical initial
weights** (the seed is reset before `create_model`), on the same mini-batches, for `EPOCHS` epochs. Every run goes to
W&B as a separate run in the group `week03-optimizers`, so the loss curves can be compared on the project page.
Optimizers whose verification cell did not print `OK` are skipped.

The learning rates below were chosen so that all four optimizers converge within the default 500 epochs on this
problem. Feel free to experiment (for example, the "textbook" `lr=0.001` for RMSprop and Adam is noticeably slower
here), but log every experiment as its own W&B run and keep the default values in the notebook you hand in.

In [ ]:
def make_optimizers() -> dict[str, Optimizer]:
    return {
        "SGD": SGD(lr=0.01),
        "SGDMomentum": SGDMomentum(lr=0.01, beta=0.9),
        "RMSprop": RMSprop(lr=0.01, alpha=0.9),
        "Adam": Adam(lr=0.01, beta1=0.9, beta2=0.999),
    }


criterion = BCELoss()
results: dict[str, list[float]] = {}
trained: dict[str, Model] = {}

for name, optimizer in make_optimizers().items():
    if not VERIFIED.get(name, False):
        print(f"{name:12s} skipped: its verification cell has not printed OK yet")
        continue
    torch.manual_seed(SEED)                       # identical initial weights for every optimizer
    model = create_model()
    run = wandb.init(
        project=WANDB_PROJECT,
        group=WANDB_GROUP,
        name=f"week03-{name.lower()}",
        config={"optimizer": name, **optimizer.hyperparameters(), "epochs": EPOCHS, "batch_size": BATCH_SIZE,
                "m": X.shape[1], "noise": NOISE, "architecture": "2-3-4-5-1, tanh hidden, sigmoid output"},
    )
    results[name] = fit(model, optimizer, dataset, criterion, EPOCHS, wandb_run=run, desc=name)
    predictions = (model(X) > 0.5).to(DTYPE)
    accuracy = (predictions == Y).to(DTYPE).mean().item()
    run.summary["train/final_loss"] = results[name][-1]
    run.summary["train/accuracy"] = accuracy
    run.finish()
    trained[name] = model
    print(f"{name:12s} {optimizer!s:55s} final loss = {results[name][-1]:.4f}   train accuracy = {accuracy:.3f}")

missing = [name for name in make_optimizers() if name not in results]
assert not missing, f"not every optimizer passed its torch.optim check, skipped: {missing}"

In [ ]:
plt.figure(figsize=(8, 5))
for name, losses in results.items():
    plt.plot(losses, label=name)
plt.xlabel("epoch")
plt.ylabel("training loss (BCE)")
plt.title("Same model, same data, same initial weights: only the optimizer differs")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
best = min(results, key=lambda name: results[name][-1])
print(f"lowest final loss: {best} ({results[best][-1]:.4f})")
draw_decision_boundary(X, Y, trained[best], title=f"{best}: decision boundary after {EPOCHS} epochs")

## 5. Check your understanding

Answer briefly in the markdown cells (double-click to edit). Look at your loss curves (here or in W&B) before you
answer 5.1.

**5.1 Rank the four optimizers by the training loss they reach after 500 epochs and by how fast they get below,
say, 0.4. Is this ranking a property of the optimizers alone?**

_Your answer here._

**5.2 Why does momentum help on a loss surface like this one?**

_Your answer here._

**5.3 What does `eps` prevent in RMSprop and Adam? What would happen without it?**

_Your answer here._

**5.4 What would happen to Adam without the bias correction in the first steps? Work it out for the first step
with $\beta_1 = 0.9$, $\beta_2 = 0.999$.**

_Your answer here._

**5.5 All four optimizers use `lr = 0.01` above. Why is "the same learning rate" not really comparable across
optimizers?**

_Your answer here._